# 01 · Build SIH dataset (real voice-clone spoofs)


# 01 · VoiceShield SIH — Build a large SIH-compliant dataset (REAL voice cloning)

> What this notebook does:
>  1. Downloads real Indian-language human speech — **FLEURS (CC-BY-4.0)**,
>     optionally **Mozilla Common Voice (CC-BY-4.0; CC0 ≤ v13)** for
>     hi/en/kn/ta/te/ml/etc.
>  2. Generates **REAL A.I. voice impersonation**: the *same human speakers*
>     are voice-cloned with **Coqui XTTS-v2** (open, CPML-1.0) that repeat
>     attacker-style phrases. Optional **RVC** / **FreeVC** conversion.
>  3. Adds plain-TTS (edge-tts) as a *minority* auxiliary attack class.
>  4. Writes balanced train/val CSVs + optionally pushes to the HuggingFace Hub.
>
> Why: SIH problem statement = real-time A.I. voice + voice-impersonation
> detection. A model trained only on plain TTS misses the real attack:
> a cloned voice of the actual speaker. Cloud cloning = real impersonation.

## 1 · GPU + setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q datasets[audio] soundfile librosa gtts torchaudio
# Coqui XTTS-v2 voice cloning (REAL impersonation). The original "TTS" package
# is unmaintained and has NO Python>=3.12 wheels; use the maintained fork
# "coqui-tts" (same API: `from TTS.api import TTS`).
!pip install -q coqui-tts
# Optional extras (uncomment to enable):
# !pip install -q edge-tts nest_asyncio     # extra aux TTS flavour (may be rate-limited)
# !pip install -q rvc-python                # optional RVC conversion

In [ ]:
import os, numpy as np, pandas as pd

# Drive is OPTIONAL. If mounting fails (auth popup not accepted), we fall back
# to Colab local disk and you can still push the dataset to HF Hub.
DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
except Exception as exc:  # noqa: BLE001
    print("Drive mount failed — continuing on local disk:", exc)

ROOT = "/content/drive/MyDrive/voiceshield_cloud" if DRIVE_OK else "/content/vs_data"
AUDIO_DIR = os.path.join(ROOT, "audio")
os.makedirs(AUDIO_DIR, exist_ok=True)
print("DRIVE_OK =", DRIVE_OK)
print("ROOT     =", ROOT)

In [ ]:
# Optional: HF token (Write rights) to push the dataset to HuggingFace Hub.
# In Colab: pick 🔑 "Secrets" in the left sidebar → Add new secret → name "HF_TOKEN".
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN", "")
except Exception as exc:  # noqa: BLE001
    print("No Colab secret HF_TOKEN (optional):", exc)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ.setdefault("COQUI_TOS_AGREED", "1")  # silent auto-accept XTTS licence
print("HF_TOKEN loaded:", bool(HF_TOKEN))

## 2 · Canonical dataset builder

In [ ]:
# Embedded copy of backend/training/cloud_dataset.py — no GitHub needed.
import base64
CLOUD_DATASET_B64 = "..."
open("/content/cloud_dataset.py", "wb").write(base64.b64decode(CLOUD_DATASET_B64))

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("cloud_dataset", "/content/cloud_dataset.py")
cd   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(cd)
print("cloud_dataset loaded:", hasattr(cd, "fetch_fleurs"))

## 3 · REAL human speech (bonafide)

In [ ]:
LANGS = ["hindi", "english", "kannada", "tamil", "telugu", "marathi", "malayalam"]

# CC-BY-4.0 real Indian-language speech
bona = cd.fetch_fleurs(AUDIO_DIR, per_language=250, languages=LANGS)
print("FLEURS bonafide:", len(bona))
# Optional: CC0 extra real voices
# bona2 = cd.fetch_commonvoice(AUDIO_DIR, per_language=150, languages=LANGS)
# bona  = pd.concat([bona, bona2], ignore_index=True)
print("bonafide per lang:\n", bona.language.value_counts())

## 4 · REAL A.I. voice impersonation (clone the actual speakers)

In [ ]:
# XTTS-v2 clones EVERY bona-fide hindi/english speaker → speaks attacker text
# in their cloned voice. This is the "voice impersonation" the SIH problem asks about.
spoof_xtts = cd.generate_xtts_spoofs(bona, AUDIO_DIR, per_speaker=3)
print("XTTS voice-clone spoofs:", len(spoof_xtts))

In [ ]:
# Optional: RVC conversion clones (MIT) — put rvc-model.pth + index in Drive
# spoof_rvc = cd.generate_rvc_spoofs(bona, AUDIO_DIR,
#     "/content/drive/MyDrive/rvc_model", target_voice_id=0, per_speaker=2)

# Optional: FreeVC zero-shot conversion (MIT)
# spoof_fvc = cd.generate_freevc_spoofs(bona, AUDIO_DIR,
#                                       "/content/drive/MyDrive/freevc/pretrained_v2")

## 5 · Auxiliary pure-TTS attack class (minority)

In [ ]:
# gTTS (Google Translate TTS) — synchronous, reliable from Colab, covers every
# Indian language. Edge-tts (Microsoft) is optional and may fail from datacenter IPs.
spoof_gtts = cd.generate_gtsspoofs(AUDIO_DIR, per_language=30, languages=LANGS)
print("gTTS aux spoofs:", len(spoof_gtts))

# Uncomment to add edge-tts flavour (may produce 0 on Colab if Microsoft throttles)
# spoof_edge = cd.generate_edge_spoofs(AUDIO_DIR, per_language=30, languages=LANGS)
# spoof_tts  = pd.concat([spoof_gtts, spoof_edge], ignore_index=True) if len(spoof_edge) else spoof_gtts
spoof_tts = spoof_gtts

## 6 · Balanced split + archive

In [ ]:
# `build_and_split` writes train.csv/val.csv to the module DATA_DIR; point it at ROOT
cd.DATA_DIR = ROOT
train, val = cd.build_and_split(bona, [spoof_xtts, spoof_tts], val_frac=0.15)
print("train:", len(train), " val:", len(val))
print(train.label.value_counts())
print(val.label.value_counts())

In [ ]:
# Persist to ROOT (Drive if mounted; local disk otherwise). Notebooks 02/03
# reuse these CSVs if DRIVE_OK, or you push to HF Hub in the next cell.
os.makedirs(ROOT, exist_ok=True)
train.to_csv(os.path.join(ROOT, "train.csv"), index=False)
val.to_csv(os.path.join(ROOT, "val.csv"), index=False)
print("Saved CSVs →", ROOT)
print("(audio dir →", AUDIO_DIR, ")")

In [ ]:
# Optional: push CSV to HF Hub so Kaggle/Colab can reload it by URL.
# Requires the HF_TOKEN secret (Write rights). Private repo so your data stays yours.
if HF_TOKEN and HF_TOKEN.startswith("hf_"):
    from datasets import Dataset, DatasetDict, Audio, ClassLabel, Features, Value

    def to_hf(df):
        feats = Features({"audio": Audio(sampling_rate=16000),
                          "label": ClassLabel(names=["bonafide","spoof"]),
                          "language": Value("string")})
        d = df.copy()
        d["label"] = (d.label == "spoof").astype("int8")
        d = d.rename(columns={"audio_path": "audio"})
        return Dataset.from_pandas(d[["audio","label","language"]], features=feats)

    ds = DatasetDict({"train": to_hf(train), "val": to_hf(val)})
    ds.push_to_hub("VAIVE/voiceshield-sih", private=True)
    print("Pushed dataset → VAIVE/voiceshield-sih  (use in notebooks 02/03 via load_dataset)")
else:
    print("Skipped HF push (no valid HF_TOKEN). If DRIVE_OK=False, keep this Colab session "
          "alive and run notebook 02 in the SAME session, or set the HF_TOKEN secret and rerun.")

In [ ]:
print("NEXT → open colab_02_train_wav2vec2.ipynb")